# Using the BLADE Pipeline to Submit a Data Science Task to LMs

This notebook builds on `example_prompt.ipynb` by using the new
`PromptGenerator` class to customize the prompt to the language model,
allowing us to apply perturbations. This class should be built out further
to consist of more flexible and easy-to-define perturbations.

## Setup

In [1]:
# imports
import yaml
from stat_genie.lm_runner.pipeline.config import MultiRunConfig
from stat_genie.lm_runner.pipeline.multirun import multirun_llm
from stat_genie.lm_runner.pipeline.prompt import PromptGenerator

## Prompt #1

In [2]:
# set up the first prompt
system_prompt = """You are an AI Data Analysis Assistant who is an expert at \
writing an end-to-end scientific analysis given a research question and a dataset. \
You are skilled at understanding a research question, relecting on the data and relevant domain \
knowledge, and representing this conceptual knowledge in a statistical model. \
Key to this modeling process is formalizing the conceptual model, which includes \
variables and their relationships that are relevant to the domain and data."""


instruction_prompt = """<Instruction> 
Given the research question, dataset \
formulate the conceptual model and write an analysis including all necessary \
data transformations and a statistical model to answer the research question. 
</Instruction>

<Format Instructions>
You will return 3 things:
1. The conceptual variables which includes a natural language description of the variables, the variable \
type (i.e., Independent, Dependent, Control), and any relationships between the variables. Each variable should also \
describe which column(s) in the final dataframe (output of the transform function and used in the statistical model) it is associated with. \
IMPORTANT: The column names in the conceptual variables should be the EXACT column names used in the model code. \
    
2. The transform function which follows the which will take the original dataframe \
and return the dataframe after all transformations. \
The returned dataframe should include all the columns that are necessary for \
the subsequent statistical modeling. \
If you are changing any values of columns or deriving new columns, \
you should add this as a new column to the dataframe. \
    
3. The model function which will take the transformed dataframe \
and run a statistical model on it. The model function should return the results of the model.

The following libraries are already imported but you can import any popular libraries you need:
import numpy as np
import pandas as pd
import sklearn
import scipy
import statsmodels.api as sm
import matplotlib.pyplot as plt

Here is the code template for the transform function:
```python
def transform(df: pd.DataFrame) -> pd.DataFrame:
    # Your code here
    return df
```
Here is the code template for the model function:
```python
def model(df: pd.DataFrame) -> Any:
    # Your code here
    return results
```

Please return the conceptual variables, the transform function, and the model function in the format specified below:
{format_instructions}
</Format Instructions>
"""

example = """<Example>
Research Question: {research_question_ex}
Dataset Schema: {dataset_schema_ex}
Result: {result_ex}
</Example>
"""

post_fix = """Research Question: {research_question}
Dataset Schema: {dataset_schema}
Result: """

In [3]:
prompt_generator = PromptGenerator(
    system_prompt=system_prompt,
    instruction_prompt=instruction_prompt,
    post_fix=post_fix,
    example=example,
)

In [4]:
### set config parameters

# set up config object
llm_provider = "openai"
llm_model = "gpt-3.5-turbo"
llm_config = yaml.safe_load(open("../../../blade-demos/openai_config.yml"))
llm_config["provider"] = llm_provider
llm_config["model"] = llm_model
llm_eval_config = llm_config

# set rest of parameters
output_dir = "results_custom_prompt_1"
run_dataset = "hurricane"
use_agent = False
use_data_desc = True
num_runs=5
use_code_cache=False

In [5]:
# the MultiRunConfig object is how BLADE standardizes experiment configuration
single_run_config = MultiRunConfig(llm=llm_config,
                llm_eval=llm_eval_config,
                output_dir=output_dir,
                run_dataset=run_dataset,
                use_agent=use_agent,
                use_data_desc=use_data_desc,
                num_runs=num_runs,
                use_code_cache=use_code_cache,
)

There are a couple things to note with the cell below. We see in the last logger
line that:

`[2025-10-09 10:05:16.73][llm.py:109 - blade_bench.llms.llm:generate][PROMPT] Sending prompt from <class 'stat_genie.lm_runner.pipeline.gen_analysis.GenAnalysisLM'>`.

This shows that the prompt wording that it is submitting comes from
`lm_runner/pipeline/gen_analysis.py`. We can therefore change the variables
defined in that file to change the prompt (or ideally set up a function that
allows for user-input perturbations of some sort).

The second thing to note is that for some reason, it is not satisfied with
some config-related thing. When I (Zach) run the cell below, I get told that my
"LLM_CONFIG_PATH environment variable is not set to a valid config file" and it
thus uses the default config file, possibly overwriting the custom config I made
above. Weird! I will look into this later.

In [6]:
# run the experiment
multirun_llm(single_run_config, prompt_generator)

[2025-10-10 03:00:36.68][config_load.py:27 - blade_bench.llms.config_load:load_config][INFO] Info: LLM_CONFIG_PATH environment variable is not set to a valid config file. Using default config file at '/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/blade_bench/conf/llm_config.yml'.
[2025-10-10 03:00:36.75][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/blade_bench/conf/llm_config.yml'.
[2025-10-10 03:00:37.27][config_load.py:27 - blade_bench.llms.config_load:load_config][INFO] Info: LLM_CONFIG_PATH environment variable is not set to a valid config file. Using default config file at '/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/blade_bench/conf/llm_config.yml'.
[2025-10-10 03:00:37.31][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/

## Prompt #2

In [7]:
# set up the first prompt
system_prompt = """You are an AI Data Analysis Assistant who is an expert at \
writing an end-to-end scientific analysis given a research question and a dataset. \
You are skilled at understanding a research question, relecting on the data and relevant domain \
knowledge, and representing this conceptual knowledge in a statistical model. \
Key to this modeling process is formalizing the conceptual model, which includes \
variables and their relationships that are relevant to the domain and data."""


instruction_prompt = """<Instruction> 
Given the research question, dataset \
formulate the conceptual model and write an analysis including all necessary \
data transformations and a statistical model to answer the research question. 
</Instruction>

<Format Instructions>
You will return 2 things:

1. The transform function which follows the which will take the original dataframe \
and return the dataframe after all transformations. \
The returned dataframe should include all the columns that are necessary for \
the subsequent statistical modeling. \
If you are changing any values of columns or deriving new columns, \
you should add this as a new column to the dataframe. \
    
2. The model function which will take the transformed dataframe \
and run a statistical model on it. The model function should return the results of the model. \
The results of the model should be a single number (e.g. model coefficient, R-squared value, p-value, etc.) \
that summarizes the key finding/conclusion from the model. \

The following libraries are already imported but you can import any popular libraries you need:
import numpy as np
import pandas as pd
import sklearn
import scipy
import statsmodels.api as sm
import matplotlib.pyplot as plt

Here is the code template for the transform function:
```python
def transform(df: pd.DataFrame) -> pd.DataFrame:
    # Your code here
    return df
```
Here is the code template for the model function:
```python
def model(df: pd.DataFrame) -> Any:
    # Your code here
    return result
```

Please return the conceptual variables, the transform function, and the model function in the format specified below:
{format_instructions}
</Format Instructions>
"""

example = """<Example>
Research Question: {research_question_ex}
Dataset Schema: {dataset_schema_ex}
Result: {result_ex}
</Example>
"""

post_fix = """Research Question: {research_question}
Dataset Schema: {dataset_schema}
Result: """

In [8]:
prompt_generator = PromptGenerator(
    system_prompt=system_prompt,
    instruction_prompt=instruction_prompt,
    post_fix=post_fix,
    example=example,
)

In [9]:
### set config parameters

# set up config object
llm_provider = "openai"
llm_model = "gpt-3.5-turbo"
llm_config = yaml.safe_load(open("../../../blade-demos/openai_config.yml"))
llm_config["provider"] = llm_provider
llm_config["model"] = llm_model
llm_eval_config = llm_config

# set rest of parameters
output_dir = "results_custom_prompt_2"
run_dataset = "hurricane"
use_agent = False
use_data_desc = True
num_runs=5
use_code_cache=False

In [10]:
# the MultiRunConfig object is how BLADE standardizes experiment configuration
single_run_config = MultiRunConfig(llm=llm_config,
                llm_eval=llm_eval_config,
                output_dir=output_dir,
                run_dataset=run_dataset,
                use_agent=use_agent,
                use_data_desc=use_data_desc,
                num_runs=num_runs,
                use_code_cache=use_code_cache,
)

There are a couple things to note with the cell below. We see in the last logger
line that:

`[2025-10-09 10:05:16.73][llm.py:109 - blade_bench.llms.llm:generate][PROMPT] Sending prompt from <class 'stat_genie.lm_runner.pipeline.gen_analysis.GenAnalysisLM'>`.

This shows that the prompt wording that it is submitting comes from
`lm_runner/pipeline/gen_analysis.py`. We can therefore change the variables
defined in that file to change the prompt (or ideally set up a function that
allows for user-input perturbations of some sort).

The second thing to note is that for some reason, it is not satisfied with
some config-related thing. When I (Zach) run the cell below, I get told that my
"LLM_CONFIG_PATH environment variable is not set to a valid config file" and it
thus uses the default config file, possibly overwriting the custom config I made
above. Weird! I will look into this later.

In [11]:
# run the experiment
multirun_llm(single_run_config, prompt_generator)

[2025-10-10 03:01:01.26][config_load.py:27 - blade_bench.llms.config_load:load_config][INFO] Info: LLM_CONFIG_PATH environment variable is not set to a valid config file. Using default config file at '/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/blade_bench/conf/llm_config.yml'.
[2025-10-10 03:01:01.30][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/blade_bench/conf/llm_config.yml'.
[2025-10-10 03:01:01.49][config_load.py:27 - blade_bench.llms.config_load:load_config][INFO] Info: LLM_CONFIG_PATH environment variable is not set to a valid config file. Using default config file at '/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/blade_bench/conf/llm_config.yml'.
[2025-10-10 03:01:01.54][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/